# Class-aware NONAN healthy-alignment pilot

This pilot compares the original binary model with a healthy-only CORAL-style embedding alignment loss. NONAN contributes only known healthy examples; no frozen NONAN or RevalExo data are read. All splits are participant-disjoint.

In [1]:
from pathlib import Path
import sys, gc
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name.lower() == 'notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P = ROOT/'data'/'processed'; N = ROOT/'data'/'interim'/'nonan_gaitprint'; D = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', D)
x = np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'), np.load(P/'sint_maartenskliniek_external_windows_float32.npy')])
m = pd.concat([pd.read_csv(P/'validated_window_metadata.csv'), pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')], ignore_index=True)
m = m.loc[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y'] = m.label.eq('stroke').astype(int); m['group'] = m.participant_key.astype(str)
nx = np.load(N/'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy', mmap_mode='r'); nm = pd.read_csv(N/'candidate_healthy_enrichment_window_metadata.csv'); nm['y'] = 0; nm['group'] = nm.participant_key.astype(str)
cap_rng = np.random.default_rng(42); keep = np.concatenate([cap_rng.choice(v, min(64,len(v)),replace=False) for v in nm.groupby('group').indices.values()]); nx = np.asarray(nx[keep]); nm = nm.iloc[keep].reset_index(drop=True)
people = pd.concat([m[['group','y']].drop_duplicates().assign(source='original'), nm[['group','y']].drop_duplicates().assign(source='nonan')], ignore_index=True); people['stratum'] = people.source+'|'+people.y.astype(str)
def picks(rng, idx, n): return rng.choice(np.flatnonzero(idx), n, replace=True)
def probabilities(net, arr, mean, std):
    with torch.inference_mode(): return torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
def coral(a,b): return (a.mean(0)-b.mean(0)).pow(2).mean() + (a.var(0,unbiased=False)-b.var(0,unbiased=False)).pow(2).mean()
rows=[]; folds=StratifiedKFold(3,shuffle=True,random_state=42)
for fold,(trp,vap) in enumerate(folds.split(people,people.stratum)):
    trgroups, vagroups = set(people.iloc[trp].group), set(people.iloc[vap].group)
    bt,bv=m.group.isin(trgroups).to_numpy(),m.group.isin(vagroups).to_numpy(); nt,nv=nm.group.isin(trgroups).to_numpy(),nm.group.isin(vagroups).to_numpy()
    for mode in ['baseline','healthy_coral_0p05']:
        torch.manual_seed(4200+fold); rng=np.random.default_rng(4200+fold)
        tx = x[bt] if mode=='baseline' else np.concatenate([x[bt],nx[nt]]); mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4)
        ox=torch.from_numpy(((x[bt]-mean)/std).transpose(0,2,1).astype('float32')); oy=torch.from_numpy(m.loc[bt,'y'].to_numpy('float32')); qx=torch.from_numpy(((nx[nt]-mean)/std).transpose(0,2,1).astype('float32'))
        original_healthy=(oy.numpy()==0); original_stroke=(oy.numpy()==1); net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
        for epoch in range(8):
            net.train()
            for _ in range(90):
                ih=picks(rng,original_healthy,64); is_=picks(rng,original_stroke,64)
                if mode=='baseline': xb=torch.cat([ox[ih],ox[is_]]).to(D); yb=torch.cat([oy[ih],oy[is_]]).to(D); h=net.f(xb).flatten(1); logit=net.c(h).squeeze(1); loss=torch.nn.functional.binary_cross_entropy_with_logits(logit,yb)
                else:
                    iq=rng.integers(0,len(qx),16); xb=torch.cat([ox[ih[:56]],ox[is_[:56]],qx[iq]]).to(D); yb=torch.cat([oy[ih[:56]],oy[is_[:56]],torch.zeros(16)]).to(D); h=net.f(xb).flatten(1); logit=net.c(h).squeeze(1); ce=torch.nn.functional.binary_cross_entropy_with_logits(logit,yb,reduction='none'); weights=torch.cat([torch.ones(112),torch.full((16,),.1)]).to(D); loss=(ce*weights).sum()/weights.sum()+.05*coral(h[:56],h[112:])
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        for name,ex,em in [('original',x[bv],m.loc[bv]),('nonan_holdout',nx[nv],nm.loc[nv])]:
            p=probabilities(net,ex,mean,std); g=em.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); rows.append({'fold':fold,'mode':mode,'evaluation':name,'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p) if g.y.nunique()==2 else np.nan,'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5) if g.y.nunique()==2 else np.nan,'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean())})
        del net,opt,ox,oy,qx; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',fold,mode)
out=pd.DataFrame(rows); out.to_csv(P/'class_aware_nonan_alignment_pilot.csv',index=False); print(out.groupby(['mode','evaluation'])[['auroc','balanced_accuracy','healthy_specificity']].mean())

device: cuda


complete 0 baseline


complete 0 healthy_coral_0p05


complete 1 baseline


complete 1 healthy_coral_0p05


complete 2 baseline


complete 2 healthy_coral_0p05
                                     auroc  balanced_accuracy  \
mode               evaluation                                   
baseline           nonan_holdout       NaN                NaN   
                   original       0.953322           0.885731   
healthy_coral_0p05 nonan_holdout       NaN                NaN   
                   original       0.957272           0.848822   

                                  healthy_specificity  
mode               evaluation                          
baseline           nonan_holdout             0.987179  
                   original                  0.873016  
healthy_coral_0p05 nonan_holdout             0.987179  
                   original                  0.809524  


CORAL aligns only healthy embeddings from the candidate and existing sources; it does not force stroke and healthy distributions together. This is a pilot. Repeat with paired seeds only if it improves held-out candidate healthy specificity without degrading original-source metrics.